## Use Case Overview: 

The objective of this notebook is to predict the duration of NYC taxi trips. \
We use data related to yellow or green taxis to train a simple prediction model.

In this notebook, we use:

- Yellow taxi data from January 2021 for model training.
- Data from February 2021 to test the model (make predictions).

**Features for Model Training**: Useful variables related to the trip itself 

**Target to Predict**: The duration of the trip


### Data

Download the three Parquet files by following the `curl` instructions in the
[course README](./README.md#introduction-to-the-use-case) before running this notebook.
The files are expected in the `data` directory at the root of the repository.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
from scipy.sparse import csr_matrix
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

DATA_FOLDER = Path("../../data")


**Info:** This notebook processes more than one million rows, so some cells can take a few minutes to run.


# 1 - Load data

In [ ]:
train_df = pd.read_parquet(DATA_FOLDER / "yellow_tripdata_2021-01.parquet")


In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.isnull().sum()

# 2 - Prepare the data

## 2-1 Compute the target

The dataset contains pickup and dropoff time but not the duration itself. \
We compute the duration of a taxi trip in minutes using these two variables.

In [ ]:
def compute_target(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with trip duration expressed in minutes."""
    result = df.copy()
    trip_duration = result["tpep_dropoff_datetime"] - result["tpep_pickup_datetime"]
    result["duration"] = trip_duration.dt.total_seconds() / 60
    return result


train_df = compute_target(train_df)


We can visualise how the duration is distributed : 

In [ ]:
train_df["duration"].describe()

In [ ]:
fig = ff.create_distplot(
    [train_df["duration"].dropna()],
    ["duration"],
    show_hist=False,
    show_rug=False,
)
fig.update_layout(width=1200, height=600)
fig.update_xaxes(title="Trip duration (minutes)")
fig.update_yaxes(title="Density")
fig.show()


We notice negative durations and extreme values. We remove outliers and restrict the modelling scope to trips lasting between 1 and 60 minutes.

In [ ]:
MIN_DURATION = 1
MAX_DURATION = 60


def filter_outliers(
    df: pd.DataFrame,
    min_duration: int = MIN_DURATION,
    max_duration: int = MAX_DURATION,
) -> pd.DataFrame:
    """Return trips whose duration is within the accepted range."""
    valid_duration = df["duration"].between(min_duration, max_duration)
    return df.loc[valid_duration].copy()


train_df = filter_outliers(train_df)


In [ ]:
fig = px.histogram(
    train_df,
    x="duration",
    nbins=60,
    title="Trip duration after filtering",
    labels={"duration": "Trip duration (minutes)"},
)
fig.update_layout(yaxis_title="Number of trips", bargap=0.02)
fig.show()


## 2-2 Prepare features

### 2-2-1 Categorical features

We will encode discrete variables as strings and then proceed to extracting the features and the target in order to train two models:

In [ ]:
CATEGORICAL_COLS = ["PULocationID", "DOLocationID", "passenger_count"]


def encode_categorical_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with categorical columns encoded as strings."""
    result = df.copy()
    result[CATEGORICAL_COLS] = result[CATEGORICAL_COLS].fillna(-1).astype(int).astype(str)
    return result


train_df = encode_categorical_cols(train_df)


In [ ]:
def extract_x_y(
    df: pd.DataFrame,
    dv: DictVectorizer | None = None,
) -> tuple[csr_matrix, np.ndarray, DictVectorizer]:
    """Return vectorized features, target, and fitted vectorizer."""
    records = df[CATEGORICAL_COLS].to_dict(orient="records")

    if dv is None:
        dv = DictVectorizer()
        X = dv.fit_transform(records)
    else:
        X = dv.transform(records)

    y = df["duration"].to_numpy()
    return X, y, dv


X_train, y_train, dv = extract_x_y(train_df)


# 3 - Train model

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

In [ ]:
# ~ 2.5 minutes
rf = RandomForestRegressor(n_estimators=100, max_depth=11, max_features="sqrt", random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)


# 4 - Evaluate model

In [ ]:
def compute_metrics(y: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Compute RMSE and MAPE."""
    metrics = {
        "rmse": float(root_mean_squared_error(y, y_pred)),
        "mape": float(mean_absolute_percentage_error(y, y_pred)),
    }
    return metrics


**Metric reminder:** RMSE is expressed in minutes and penalizes large errors, while MAPE expresses the average relative error and can be misleading when observed values are close to zero; see the [scikit-learn regression metrics guide](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).


## 4-1 On train data

In [ ]:
y_pred_lr = lr.predict(X_train)
y_pred_rf = rf.predict(X_train)
compute_metrics(y_train, y_pred_lr), compute_metrics(y_train, y_pred_rf)

## 4-2 On test data

In [ ]:
test_df = pd.read_parquet(DATA_FOLDER / "yellow_tripdata_2021-02.parquet")


In [ ]:
test_df = compute_target(test_df)
test_df = filter_outliers(test_df)
test_df = encode_categorical_cols(test_df)
X_test, y_test, _ = extract_x_y(test_df, dv=dv)

In [ ]:
y_pred_test_lr = lr.predict(X_test)
y_pred_test_rf = rf.predict(X_test)
compute_metrics(y_test, y_pred_test_lr), compute_metrics(y_test, y_pred_test_rf)

# Conclusion

* What can you say about the trained models?
* If you had to improve these models, what would you do?


# Bonus - Interactive data profiling

[FG Data Profiling](https://pypi.org/project/fg-data-profiling/) (formerly YData Profiling) generates an eye-catching interactive dashboard with dataset statistics, missing values, distributions, correlations, interactions, duplicates, and data-quality alerts. The package was renamed in 2026, so this notebook uses the maintained `fg-data-profiling` package and its `data_profiling` import.

The complete report can be expensive on the full dataset. We therefore profile a reproducible 100,000-row sample. Use `minimal=True` in `ProfileReport` when you prefer a faster report with fewer computations. After installing the project dependencies with `uv sync`, run the next cell and explore the embedded dashboard. Profiling is a starting point for investigation, not a substitute for domain validation.


In [ ]:
from data_profiling import ProfileReport

PROFILE_SAMPLE_SIZE = 100_000
profile_df = pd.read_parquet(
    Path(DATA_FOLDER) / "yellow_tripdata_2021-01.parquet"
).sample(n=PROFILE_SAMPLE_SIZE, random_state=42)

profile = ProfileReport(
    profile_df,
    title="NYC Yellow Taxi - January 2021",
    explorative=True,
    vars={"num": {"chi_squared_threshold": 0}},
    correlations={
        "auto": {"calculate": False},
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
    },
    missing_diagrams={"heatmap": False},
)
profile.to_notebook_iframe()
